# Notebook 3: Pagination Tricks

## What this notebook covers

Two situations make basic page-by-page fetching from Notebook 1 inadequate:

1. **More than 1,000 records.** The listings endpoint silently truncates results at 1,000. Areas like South Yarra or the inner-city ring have far more than 1,000 listings for any multi-year window. This notebook solves this by splitting the retrieval into overlapping date windows.

2. **Large multi-suburb extractions.** Fetching 50 suburbs in a single run can take 20 to 30 minutes and can consume significant number of credits. If anything goes wrong mid-run, all progress is lost without a checkpoint system.

This notebook covers both techniques and shows how to combine them into a robust extraction loop.

**By the end** you will have run a complete multi-suburb extraction that can survive interruptions without wasting credits.

---

## Before You Start

**Complete Notebooks 0 and 1 before this one.** Notebook 3 builds on the probe and pagination patterns from Notebooks 0 and 1, and assumes you are already comfortable with the listings endpoint. Notebook 2 (suburb statistics) is independent and can be completed before or after this notebook.

**If you have already completed Notebooks 0 and 1**, your packages, `.env` file, and connection are already set up.

---
## Key Terms

**Cursor:** A marker that tracks your position in a large dataset. When you cannot page past position 1,000, you move the cursor (your starting date) forward and start paging again from the new position.

**Cursor advancement:** The technique of repeatedly advancing the `listedSince` date to step through a dataset that exceeds 1,000 records.

**Deduplication:** Removing duplicate records. Some listings appear in two consecutive cursor windows because they fall right on the boundary date. Storing records in a dictionary keyed by listing ID removes them automatically.

**Checkpoint:** A saved record of how far an operation has progressed, like hitting Save in a video game. If something fails, you restart from the last save point, not from the beginning.

**Resume:** Continuing an interrupted operation by loading a checkpoint and skipping the steps already completed.

**KeyboardInterrupt:** What Python raises when you press the Stop button in Jupyter (or Ctrl+C in a terminal). We catch this to save progress before stopping.

**CSV file:** A plain-text spreadsheet format that Excel and other tools can open.

---

## Part 1: The 1,000-Record Cap

The listings endpoint returns at most **1,000 records per request**. If your query matches 4,200 records, you get the first 1,000. The remaining 3,200 are silently dropped, with no error and no warning.

Notebook 1 introduced the density probe: a 1-credit check that tells you the total count before you download anything. If that count is over 1,000, simple pagination cannot retrieve the full dataset. This section shows you how to handle it.

### Credit cost per suburb

Each suburb requires one probe credit, then one credit per page of results (up to 200 records per page):

- 1–200 records → **2 credits** (1 probe + 1 page)
- 201–400 records → **3 credits** (1 probe + 2 pages)
- 401–600 records → **4 credits** (1 probe + 3 pages)
- and so on

Most inner-city suburbs have several hundred to a few thousand records, so **3–15 credits per suburb** is a typical range. The 10-suburb run at the end of this notebook spent 218 credits in total.

### Why checkpoints matter

If a 50-suburb run crashes halfway through and you have no checkpoint, you must start over from suburb 1 — spending all those credits again. With a checkpoint, the completed suburbs are skipped and only the remaining ones are fetched.

| Scenario | Credits spent | Credits wasted |
|---|---|---|
| 50-suburb run, no failure | ~150–500 depending on suburb sizes | 0 |
| Crash at suburb 35, restart **without** checkpoint | Double the above | ~105+ (suburbs 1–35 fetched twice) |
| Crash at suburb 35, restart **with** checkpoint | ~original cost + suburbs 36–50 only | 0 |

The monthly credit limit is 1,000. A single failed un-checkpointed run can waste more than 10% of your monthly budget before you collect any usable data.

---
## Setup: Check Packages

In [ ]:
missing = []
for pkg in ['requests', 'pandas', 'dotenv']:
    try:
        __import__(pkg)
    except ImportError:
        missing.append(pkg if pkg != 'dotenv' else 'python-dotenv')

if missing:
    print('Missing packages -- run in terminal:')
    print(f'  pip install {" ".join(missing)}')
else:
    print('All packages found. Setup OK.')

## Setup: Connect to the API and Create the Checkpoint Folder

The cell below also creates a `checkpoints/` folder inside `phase-2/` where
the progress files will be saved. If the folder already exists, nothing happens.

In [ ]:
import sys
import json
from pathlib import Path
from datetime import datetime
import pandas as pd

if '.' not in sys.path:
    sys.path.insert(0, '.')

from utils import PROXY_BASE, APICallTracker, probe_count

tracker = APICallTracker()

# Checkpoints are written to a "checkpoints" folder next to this notebook.
# To use a different folder, give the full path with forward slashes (/) or
# doubled backslashes (\\), since a pasted Windows path with single
# backslashes may not work.
CHECKPOINT_DIR = Path('checkpoints')
CHECKPOINT_DIR.mkdir(exist_ok=True)

CHECKPOINT_FILE = CHECKPOINT_DIR / 'extraction_checkpoint.json'

import os
print(f'Connected to: {PROXY_BASE}')
print(f'Account:      {os.getenv("AURIN_USERNAME", "(not found -- check .env)")}')
print(f'Checkpoint folder: {CHECKPOINT_DIR.resolve()}')

---

## Cursor Advancement

When the density probe shows more than 1,000 records, cursor advancement splits the retrieval into date windows.

Each window fetches up to 1,000 records sorted oldest to newest. When the cap is hit, `listedSince` advances to the listing date of the last record in that batch and the fetch restarts from there.

The cursor lands on the last record's date, not the day after, because multiple listings can share the same date. Any listings dated the same day as the 1,000th record that did not fit in the batch would be missed if the cursor skipped ahead by one day. Since we cannot see past the 1,000th record, the last known date is the only safe position.

This means some records near the boundary will be fetched twice. That is by design: results are stored in a dictionary keyed by listing ID, so a duplicate simply overwrites the original and the final dataset stays clean.

We will demonstrate this on South Yarra 3141 from 2019 onwards, which has well over 1,000 sold listings.

### Step 1: Confirm the record count

In [ ]:
# Define the South Yarra query for the cursor demonstration
south_yarra_payload = {
    'listingType': 'Sold',
    'listedSince': '2019-01-01',
    'locations': [{
        'state': 'VIC',
        'suburb': 'South Yarra',
        'postCode': '3141',
        'includeSurroundingSuburbs': False,
    }],
}

# Run a density probe to confirm the count (1 credit)
sy_count = probe_count(south_yarra_payload, tracker)
tracker.checkpoint('South Yarra probe')

print(f'South Yarra 3141 (from 2019): {sy_count:,} records')
print(f'Exceeds 1,000-record limit:   {sy_count is not None and sy_count > 1000}')
print()
print('Simple pagination would silently miss everything above record 1,000.')
print('Cursor advancement retrieves the complete dataset.')

### Step 2: Define the cursor function

The `fetch_all_cursor` function below implements cursor advancement. The comments walk through each stage.

In [ ]:
def fetch_all_cursor(base_payload, start_date, page_size=200):
    """Retrieve ALL listings for a query that may exceed 1,000 results.

    Uses cursor advancement: repeatedly advances listedSince to step through
    the full dataset in chunks of up to 1,000. Deduplicates by listing ID.

    base_payload: the search criteria (suburb, listing type) WITHOUT listedSince.
    start_date:   the earliest listing date to include (YYYY-MM-DD format).
    page_size:    records per page (max 200).

    Returns: a deduplicated list of all listing dicts.
    """
    all_listings = {}
    current_since = start_date
    prev_cursor = None
    run = 0

    while True:
        run += 1
        page = 1
        batch = []
        hit_limit = False

        while True:
            payload = {
                **base_payload,
                'listedSince': current_since,
                'sort': {'sortKey': 'DateListed', 'direction': 'Ascending'},
                'pageSize': page_size,
                'pageNumber': page,
            }
            r = tracker.post(
                f'{PROXY_BASE}/v1/listings/residential/_search',
                json_body=payload,
            )

            if r.status_code == 400 and 'Cannot page beyond 1000' in r.text:
                hit_limit = True
                break

            if r.status_code != 200:
                print(f'  Run {run}, page {page}: unexpected HTTP {r.status_code}')
                return list(all_listings.values())

            groups = r.json()
            for g in groups:
                raw = g.get('listings') or g.get('listing')
                if isinstance(raw, list):
                    batch.extend(raw)
                elif isinstance(raw, dict):
                    batch.append(raw)

            actual_page_size = int(r.headers.get('x-pagination-pagesize', page_size))
            if len(groups) < actual_page_size:
                break
            page += 1

        before = len(all_listings)
        for item in batch:
            if item.get('id'):
                all_listings[item['id']] = item
        print(f'  Cursor run {run}: +{len(all_listings) - before:>5} new | total: {len(all_listings):>6}')

        if not hit_limit or not batch:
            break

        dates = [item['dateListed'][:10] for item in batch if item.get('dateListed')]
        if not dates:
            break

        cursor = max(dates)

        if cursor == prev_cursor:
            print(f'  Cursor stuck at {cursor} -- stopping. Some records near this date may be missed.')
            break

        prev_cursor = cursor
        current_since = cursor

    return list(all_listings.values())


print('fetch_all_cursor defined.')

### Step 3: Fetch all South Yarra listings

The cell below runs the full extraction. Each cursor window fetches up to 1,000 records across up to 5 pages of 200 records each, so **one cursor window costs up to 5 credits**. The total depends on how many records match the query.

**Expected output:** Several lines like "Cursor run 1: +1,000 new | total: 1,000", continuing until a final run with fewer than 1,000 new records.

In [ ]:
print('Fetching South Yarra 3141 (cursor advancement)...')
south_yarra_raw = fetch_all_cursor(south_yarra_payload, start_date='2019-01-01')
tracker.checkpoint('South Yarra cursor fetch')

# Build a DataFrame from the results
df_sy = pd.DataFrame([
    {
        'id':            item.get('id'),
        'suburb':        (item.get('propertyDetails') or {}).get('suburb'),
        'property_type': (item.get('propertyDetails') or {}).get('propertyType'),
        'bedrooms':      (item.get('propertyDetails') or {}).get('bedrooms'),
        'date_listed':   (item.get('dateListed') or '')[:10],
        'sold_price':    (
            (item.get('soldData') or {}).get('soldPrice')
            or (item.get('priceDetails') or {}).get('price')
        ),
    }
    for item in south_yarra_raw
])

print(f'\nSouth Yarra: {len(df_sy):,} records fetched.')
df_sy.head()

### Step 4: Validate the results

Compare the cursor fetch count against the probe count from Step 1. They should be close. A small difference (a few records) is normal because new listings can appear between the probe and the full fetch. A large difference (more than 10%) suggests the cursor may have got stuck or the date window was misconfigured.

In [ ]:
print(f'Probe count:          {sy_count:>6,}')
print(f'Cursor fetched count: {len(df_sy):>6,}')
print(f'Difference:           {abs(sy_count - len(df_sy)):>6,}  (a small number is expected)')
print()

duplicates = df_sy['id'].duplicated().sum()
print(f'Duplicate listing IDs: {duplicates}  (expected: 0)')
print()

print(f'Date range covered:')
print(f'  Earliest listing: {df_sy["date_listed"].min()}')
print(f'  Latest listing:   {df_sy["date_listed"].max()}')

---

## Part 2: Checkpoint Recovery

The cursor advancement above retrieves all listings for one suburb. For a large study covering 50 suburbs, cursor advancement alone is not enough: if the run fails midway, you lose all progress and must restart from scratch, spending all those credits again.

The checkpoint pattern saves your progress to disk after each suburb. On a restart, suburbs already in the checkpoint are skipped without spending any credits. The sections below define the save/load functions, wrap cursor advancement into a per-suburb fetcher, and combine both into a checkpoint-aware extraction loop.

---
## What the Checkpoint File Contains

The checkpoint file is a JSON file with three pieces of information:

1. **`completed`:** a list of suburb labels (e.g. "Carlton 3053") that have been
   fully downloaded. When resuming, any suburb in this list is skipped.

2. **`results`:** all listing records collected so far, stored as a dictionary
   keyed by listing ID. Using a dictionary ensures that if a suburb is accidentally
   fetched twice (e.g. because it was in progress when a crash occurred), each
   listing appears only once.

3. **`last_updated`:** the date and time the checkpoint was last written.

An example checkpoint file looks like this:

```json
{
  "completed": ["Carlton 3053", "Fitzroy 3065"],
  "results": {"123456": {"id": "123456", ...}, "789012": {"id": "789012", ...}},
  "last_updated": "2026-05-13T14:22:00"
}
```

### Define the save and load functions

The `save_checkpoint()` function writes the current progress to the checkpoint file.
The `load_checkpoint()` function reads it back. Both are simple: they just convert
Python data to/from a JSON text file.

Read the comments in the cell below for a line-by-line explanation.

In [ ]:
def save_checkpoint(path, completed, results_dict):
    """Save extraction progress to a JSON file.

    path:         the file path to write to (a pathlib.Path object).
    completed:    list of suburb labels that are fully done.
    results_dict: dict mapping listing ID (str) to listing record (dict).
    """
    # Build the data structure to save
    payload = {
        'completed':    completed,
        'results':      results_dict,
        'last_updated': datetime.now().isoformat(timespec='seconds'),
    }
    # json.dump converts the Python dictionary into a text file
    with open(path, 'w') as f:
        json.dump(payload, f)
    print(f'    [saved] {len(completed)} suburbs done, {len(results_dict):,} listings.')


def load_checkpoint(path):
    """Load extraction progress from a JSON file.

    If no checkpoint file exists (first run), returns empty placeholders
    so the extraction loop can start fresh.

    Returns: (completed list, results dict).
    """
    if path.exists():  # if a checkpoint file is on disk, load it
        with open(path) as f:
            data = json.load(f)  # json.load converts the text file back into a Python dictionary
        completed = data.get('completed', [])
        results   = data.get('results',   {})
        print(f'Checkpoint loaded: {len(completed)} suburbs already done, '
              f'{len(results):,} listings.')
        print(f'Last saved: {data.get("last_updated", "unknown")}')
        return completed, results
    else:  # no checkpoint file -- start fresh
        print('No checkpoint file found. Starting from scratch.')
        return [], {}  # empty list and empty dictionary


print('save_checkpoint and load_checkpoint defined.')

### Confirm the round-trip works

Before using these functions for real data, the cell below saves a small test
checkpoint and loads it back. You should see the same data you saved, confirming
that the save and load functions are working correctly.

In [ ]:
# Save a tiny test checkpoint
test_file = CHECKPOINT_DIR / 'test_checkpoint.json'
save_checkpoint(
    test_file,
    completed=['Carlton 3053'],
    results_dict={'99999': {'id': '99999', 'suburb': 'Carlton'}}
)

# Load it back
loaded_completed, loaded_results = load_checkpoint(test_file)

print()
print(f'Completed suburbs loaded: {loaded_completed}')
print(f'Listing IDs loaded:       {list(loaded_results.keys())}')

# Remove the test file -- it was just for demonstration
test_file.unlink()
print()
print('Test complete. Test file removed.')

---
## The Extraction Functions

The two functions below handle the actual API fetching. You do not need to understand the internals — just know what each one does:

- **`fetch_suburb_listings(suburb, state, postcode, start_date)`:** downloads all sold listings for one suburb from `start_date` onwards. Uses the cursor advancement defined above, so it works correctly even for suburbs with more than 1,000 listings. Returns a dictionary mapping listing ID to listing record.

- **`extract_fields(item)`:** takes one raw listing record (a complex nested dictionary) and pulls out the fields you need into a flat row, ready to put in a spreadsheet.

Run both cells to define the functions.

In [ ]:
def fetch_suburb_listings(suburb, state, postcode, start_date,
                          listing_type='Sold', page_size=200):
    """Download all listings for one suburb using cursor advancement.

    Uses the same cursor technique defined earlier in this notebook, wrapped into
    a single function call. Works for any number of records, not just under 1,000.

    Returns a dict mapping listing ID (str) to the full listing record (dict).
    """
    base_payload = {
        'listingType': listing_type,
        'locations': [{
            'state':   state,
            'suburb':  suburb,
            'postCode': postcode,
            'includeSurroundingSuburbs': False,
        }],
    }
    all_listings = {}
    current_since = start_date
    prev_cursor = None
    run = 0

    while True:
        run += 1
        page = 1
        batch = []
        hit_limit = False

        while True:
            payload = {
                **base_payload,
                'listedSince': current_since,
                'sort': {'sortKey': 'DateListed', 'direction': 'Ascending'},
                'pageSize': page_size,
                'pageNumber': page,
            }
            r = tracker.post(
                f'{PROXY_BASE}/v1/listings/residential/_search',
                json_body=payload,
            )
            if r.status_code == 400 and 'Cannot page beyond 1000' in r.text:
                hit_limit = True
                break
            if r.status_code != 200:
                print(f'      [{suburb}] HTTP {r.status_code} on run {run} page {page}')
                return all_listings
            groups = r.json()
            for g in groups:
                raw = g.get('listings') or g.get('listing')
                if isinstance(raw, list):
                    batch.extend(raw)
                elif isinstance(raw, dict):
                    batch.append(raw)
            actual_page_size = int(r.headers.get('x-pagination-pagesize', page_size))
            if len(groups) < actual_page_size:
                break
            page += 1

        for item in batch:
            if item.get('id'):
                all_listings[str(item['id'])] = item

        if not hit_limit or not batch:
            break

        dates = [item['dateListed'][:10] for item in batch if item.get('dateListed')]
        if not dates:
            break
        cursor = max(dates)
        if cursor == prev_cursor:
            break
        prev_cursor = cursor
        current_since = cursor

    return all_listings


print('fetch_suburb_listings defined.')

In [ ]:
def extract_fields(item):
    """Pull the key fields out of a raw listing record into a flat row.

    API listing records are deeply nested dictionaries. This function navigates
    that nesting and returns a flat dictionary with one value per field,
    ready to be added to a spreadsheet-style table.

    item: one listing record (a dictionary) from the API response.
    Returns: a flat dictionary with the key fields.
    """
    # Navigate the nested structure safely: if a key is missing, .get() returns {}
    # so the next .get() does not crash
    prop = item.get('propertyDetails') or {}
    sold = item.get('soldData')        or {}
    price = item.get('priceDetails')  or {}

    return {
        'listing_id':    item.get('id'),
        'suburb':        prop.get('suburb'),
        'property_type': prop.get('propertyType'),   # e.g. 'House', 'Unit'
        'bedrooms':      prop.get('bedrooms'),
        'date_listed':   (item.get('dateListed') or '')[:10],  # keep only the date (YYYY-MM-DD)
        'date_sold':     (sold.get('soldDate')   or '')[:10] or None,
        'sold_price':    sold.get('soldPrice') or price.get('price'),
        'sale_method':   sold.get('saleMethod'),  # e.g. 'Sold_Auction', 'Sold_PrivateTreaty'
    }


print('extract_fields defined.')

---
## The Checkpoint-Aware Extraction Loop

Here is what the loop does, step by step:

1. **On startup:** load the checkpoint file if it exists. If no file exists,
   start with empty progress.
2. **For each suburb:** check if it is already in the `completed` list.
   If yes, print "already done -- skipping" and move on. If no, fetch it.
3. **After each suburb:** save the updated checkpoint to disk.
4. **If interrupted:** catch the interrupt, save one final checkpoint, and
   print instructions for resuming.

### Configure your suburb list and start date

Edit the cell below to set up your actual study area. The suburb name must be
spelled exactly as it appears on the Domain website. The start date is in
`YYYY-MM-DD` format.

> **Note:** There is a start date but no end date. The extraction retrieves all
> records from `START_DATE` up to the most recent listing available at the time
> you run it. If you need to limit your study to a specific period, you will
> need to filter the resulting DataFrame by date after the extraction completes.

In [ ]:
# Edit this cell to define your study area
# Each entry is (suburb name, state code, postcode)
SUBURBS_TO_FETCH = [
    ('Carlton',     'VIC', '3053'),
    ('Fitzroy',     'VIC', '3065'),
    ('Collingwood', 'VIC', '3066'),
    ('Richmond',    'VIC', '3121'),
    ('Brunswick',   'VIC', '3056'),
    ('Northcote',   'VIC', '3070'),
    ('Abbotsford',  'VIC', '3067'),
    ('Prahran',     'VIC', '3181'),
    ('St Kilda',    'VIC', '3182'),
    ('South Yarra', 'VIC', '3141'),
]

# Only include listings where the listing date is on or after this date
START_DATE = '2022-01-01'

print(f'Suburbs to fetch: {len(SUBURBS_TO_FETCH)}')
print(f'Start date: {START_DATE}')
print()
print('Suburb list:')
for suburb, state, postcode in SUBURBS_TO_FETCH:
    print(f'  {suburb} {postcode} ({state})')

### Run the extraction

**This cell will make real API calls and spend real credits.** Before running it:

- Confirm the suburb list and start date above are correct.
- Check your remaining credit balance on the AURIN dashboard.

The cell will print progress as it goes. If you press the Stop button in Jupyter
(or close your laptop), the checkpoint is saved and you can re-run this cell to
continue from where you stopped.

In [ ]:
# Load any existing checkpoint (returns empty lists/dicts if none exists)
completed, results = load_checkpoint(CHECKPOINT_FILE)
print()

try:
    for suburb, state, postcode in SUBURBS_TO_FETCH:
        key = f'{suburb} {postcode}'  # a label for this suburb used in the checkpoint

        if key in completed:
            # This suburb is already done -- skip it without spending any credits
            print(f'  [{key}] already done -- skipping.')
            continue

        print(f'  Fetching {key}...')
        suburb_results = fetch_suburb_listings(
            suburb, state, postcode, start_date=START_DATE
        )

        # Add this suburb's listings to the running total
        results.update(suburb_results)

        # Mark this suburb as done
        completed.append(key)

        # Save progress to disk immediately after each suburb completes
        save_checkpoint(CHECKPOINT_FILE, completed, results)

        print(f'  [{key}] done: {len(suburb_results):,} listings.')

except KeyboardInterrupt:
    # This block runs if you press the Stop button in Jupyter
    print()
    print('Stopped by user. Saving progress...')
    save_checkpoint(CHECKPOINT_FILE, completed, results)
    print(f'Progress saved: {len(completed)}/{len(SUBURBS_TO_FETCH)} suburbs done.')
    print('Re-run this cell to continue from where you stopped.')

else:
    # This block runs only if no error or interrupt occurred
    tracker.checkpoint('Full extraction')
    print()
    print(f'Extraction complete: {len(completed)} suburbs, {len(results):,} listings.')

---
## Demonstration: Simulating a Failure and Recovering

The next few cells show exactly what happens when an extraction fails partway through.
We deliberately inject an error at suburb 5, inspect the checkpoint file that was saved,
and then resume.

**You do not need to run these cells in a real extraction.** They are here so you can
see the recovery process working before you rely on it for important data.

### Step 1: Run until the injected failure

In [ ]:
# DEMONSTRATION: clear the checkpoint and run until a deliberate error

# Clear any existing checkpoint so the demonstration starts from scratch
if CHECKPOINT_FILE.exists():
    CHECKPOINT_FILE.unlink()  # unlink() deletes the file
    print('Existing checkpoint cleared for demonstration.')

# Use a separate tracker so demonstration calls are counted separately
demo_tracker_original = tracker
tracker = APICallTracker()

demo_completed = []
demo_results   = {}

try:
    for i, (suburb, state, postcode) in enumerate(SUBURBS_TO_FETCH):
        key = f'{suburb} {postcode}'

        if key in demo_completed:
            continue

        # DELIBERATELY INJECT A FAILURE at the 5th suburb (index 4)
        if i == 4:
            raise RuntimeError(  # this simulates a crash or network failure
                f'Simulated failure at suburb {i+1}: {key}'
            )

        print(f'  [{i+1}/{len(SUBURBS_TO_FETCH)}] Fetching {key}...')
        suburb_results = fetch_suburb_listings(
            suburb, state, postcode, start_date=START_DATE
        )
        demo_results.update(suburb_results)
        demo_completed.append(key)
        save_checkpoint(CHECKPOINT_FILE, demo_completed, demo_results)
        print(f'  [{key}] done.')

except RuntimeError as e:
    print()
    print(f'FAILURE: {e}')
    print(f'Checkpoint contains {len(demo_completed)} completed suburbs.')
    print('In a real scenario, this would be a network error or kernel crash.')

finally:
    # Restore the original tracker
    tracker = demo_tracker_original

### Step 2: Inspect the checkpoint file

The cell below reads the checkpoint file and shows what was saved before the failure.

In [ ]:
if CHECKPOINT_FILE.exists():
    with open(CHECKPOINT_FILE) as f:
        checkpoint_data = json.load(f)

    print(f'Checkpoint file size: {CHECKPOINT_FILE.stat().st_size:,} bytes')
    print(f'Last saved:           {checkpoint_data.get("last_updated", "unknown")}')
    print(f'Completed suburbs:    {checkpoint_data["completed"]}')
    print(f'Listings saved:       {len(checkpoint_data["results"]):,}')
    print()
    print('These suburbs will be SKIPPED when we resume.')
else:
    print('No checkpoint file found.')

### Step 3: Resume

The cell below runs the same extraction loop. Suburbs already in the checkpoint
are skipped immediately with a message. Only the remaining suburbs are fetched.

In [ ]:
# Load the checkpoint from before the failure
resume_completed, resume_results = load_checkpoint(CHECKPOINT_FILE)
print()

try:
    for suburb, state, postcode in SUBURBS_TO_FETCH:
        key = f'{suburb} {postcode}'

        if key in resume_completed:
            print(f'  [{key}] already done -- skipping.')  # no credit spent
            continue

        print(f'  Fetching {key}...')
        suburb_results = fetch_suburb_listings(
            suburb, state, postcode, start_date=START_DATE
        )
        resume_results.update(suburb_results)
        resume_completed.append(key)
        save_checkpoint(CHECKPOINT_FILE, resume_completed, resume_results)
        print(f'  [{key}] done.')

except KeyboardInterrupt:
    print('Stopped. Progress saved.')

else:
    tracker.checkpoint('Resume run')
    print()
    skipped = len([k for k in [f'{s} {p}' for s, _, p in SUBURBS_TO_FETCH]
                   if k in demo_completed])
    print(f'Resume complete.')
    print(f'  Suburbs skipped (already in checkpoint): {skipped}')
    print(f'  Suburbs fetched in this run:             {len(resume_completed) - skipped}')
    print(f'  Total listings collected:                {len(resume_results):,}')

---
## Building the Final Dataset

Once all suburbs are done, load the checkpoint and build a single combined table.
This table can be saved as a CSV file that you can open in Excel, R, QGIS, or
any other analysis tool.

In [ ]:
# Load the final checkpoint
final_completed, final_results = load_checkpoint(CHECKPOINT_FILE)

if final_results:
    # Build a DataFrame: one row per listing, using extract_fields to flatten each record
    df_final = pd.DataFrame([
        extract_fields(item) for item in final_results.values()
    ])

    # Convert date columns to proper date format for easier filtering
    df_final['date_sold']   = pd.to_datetime(df_final['date_sold'],   errors='coerce')
    df_final['date_listed'] = pd.to_datetime(df_final['date_listed'], errors='coerce')

    print(f'Final dataset: {len(df_final):,} listings across {df_final["suburb"].nunique()} suburbs')
    print()
    print('Records per suburb:')
    print(df_final.groupby('suburb').size().sort_values(ascending=False).to_string())
    print()
    df_final.head(5)
else:
    print('No results in checkpoint. Run the extraction loop above first.')

### Export to CSV

A CSV file is a plain-text spreadsheet that Excel and other tools can open.
Run the cell below to save your dataset. The file will appear in the same folder
as this notebook (`training-materials/phase-2/`).

In [ ]:
# Change the filename to something meaningful for your study
# If you save somewhere other than this folder, give the full path, using
# forward slashes (/) or doubled backslashes (\\). A pasted Windows path with
# single backslashes will not work.
output_filename = 'inner_melbourne_sold_listings.csv'

if 'df_final' in dir() and len(df_final) > 0:
    # index=False means do not write the row numbers as a column
    df_final.to_csv(output_filename, index=False)
    print(f'Saved: {output_filename}')
    print(f'Rows: {len(df_final):,}')
    print(f'Columns: {list(df_final.columns)}')
    print()
    print('You can now open this file in Excel or import it into R, QGIS, or STATA.')
else:
    print('No data to save. Run the dataset-building cell above first.')

---
## What If Something Goes Wrong?

**"No checkpoint file found" on startup:**
This is normal on the first run. The loop will start from the beginning.

**A suburb returns zero listings:**
Check the suburb name and postcode spelling. The suburb name must match Domain
exactly, including capitalisation. For example, "St Kilda" not "st kilda".

**The final row count seems too low:**
Run a density probe (from Notebook 1) on a few suburbs to check the expected
counts, then compare to what is in `df_final`.

**How to start completely fresh:**
Delete the checkpoint file at `checkpoints/extraction_checkpoint.json`
and re-run the extraction loop. You can delete it from Jupyter's file browser
or from your operating system's file manager.

**Authentication failure:**
Check your `.env` file in the main `domain_api` folder.

---

## What's Next: Notebook 4

You can now handle the two most common challenges in large-scale listings extractions: areas with more than 1,000 records (cursor advancement) and runs that can be interrupted without losing progress (checkpoint recovery).

**Notebook 4** covers spatial querying. Instead of searching by suburb name, you provide a geographic boundary file (an SA2, LGA, or any custom shape) and ask for all listings whose property coordinates fall inside it. The same cursor advancement technique from this notebook applies equally to spatial queries.

→ Open `notebook-4-spatial-querying.ipynb` to continue.

---
## Credit Summary

The table below shows credits consumed in the resume run only (the second run
after the simulated failure). Compare the "Suburbs fetched in this run" count
to the total number of suburbs to confirm checkpoint skipping worked correctly.

In [ ]:
tracker.summary()